# Tourism Experience Analytics - Exploratory Data Analysis

## Comprehensive EDA for Tourism Dataset

This notebook performs thorough exploratory data analysis including:
- Data loading and inspection
- Data cleaning
- Statistical analysis
- Visualizations and insights
- Feature engineering preparation

**Author**: Tourism Analytics Team
**Date**: January 2026

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Visualization settings
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Libraries imported successfully!')

## 1. Data Loading

Loading all 9 datasets:
1. Transaction Data (visits and ratings)
2. User Data (demographics)
3. Item Data (attractions)
4. City, Country, Region, Continent Data
5. Visit Mode and Attraction Type Data

In [ ]:
# Load all datasets
transaction = pd.read_excel('../data/Transaction.xlsx')
user = pd.read_excel('../data/User.xlsx')
city = pd.read_excel('../data/City.xlsx')
continent = pd.read_excel('../data/Continent.xlsx')
country = pd.read_excel('../data/Country.xlsx')
region = pd.read_excel('../data/Region.xlsx')
item = pd.read_excel('../data/Item.xlsx')
visit_mode = pd.read_excel('../data/Mode.xlsx')
attraction_type = pd.read_excel('../data/Type.xlsx')

print('All datasets loaded!')
print(f'\nTransaction: {transaction.shape}')
print(f'User: {user.shape}')
print(f'Item: {item.shape}')
print(f'City: {city.shape}')
print(f'Continent: {continent.shape}')
print(f'Country: {country.shape}')
print(f'Region: {region.shape}')
print(f'Visit Mode: {visit_mode.shape}')
print(f'Attraction Type: {attraction_type.shape}')

## 2. Initial Data Inspection

### 2.1 Transaction Data

In [ ]:
print('='*80)
print('TRANSACTION DATA')
print('='*80)
print('\nFirst 5 rows:')
display(transaction.head())
print('\nData Info:')
print(transaction.info())
print('\nBasic Statistics:')
display(transaction.describe())
print('\nMissing Values:')
print(transaction.isnull().sum())

### 2.2 User Data

In [ ]:
print('='*80)
print('USER DATA')
print('='*80)
display(user.head())
print('\nMissing Values:')
print(user.isnull().sum())

## 3. Data Cleaning

In [ ]:
# Clean city data
city['CityName'].fillna('Unknown', inplace=True)
print('✓ Filled missing city names')

# Clean user data
user['CityId'].fillna(0, inplace=True)
user['CityId'] = user['CityId'].astype(int)
print('✓ Filled missing user CityId')

# Remove placeholder entries
city = city[city['CityId'] != 0].reset_index(drop=True)
continent = continent[continent['ContinentId'] != 0].reset_index(drop=True)
country = country[country['CountryId'] != 0].reset_index(drop=True)
region = region[region['RegionId'] != 0].reset_index(drop=True)
visit_mode = visit_mode[visit_mode['VisitModeId'] != 0].reset_index(drop=True)
print('✓ Removed placeholder entries')

print('\nData cleaning completed!')

## 4. Rating Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(transaction['Rating'], bins=5, edgecolor='black', color='skyblue')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Ratings')
axes[0].grid(axis='y', alpha=0.3)

# Bar chart
rating_counts = transaction['Rating'].value_counts().sort_index()
axes[1].bar(rating_counts.index, rating_counts.values, color='coral', edgecolor='black')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Count')
axes[1].set_title('Rating Counts')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/rating_distribution_eda.png', dpi=300, bbox_inches='tight')
plt.show()

print('Rating Statistics:')
print(transaction['Rating'].describe())

## 5. Visit Mode Analysis

In [ ]:
# Merge visit mode names
transaction_with_mode = transaction.merge(
    visit_mode.rename(columns={'VisitMode': 'VisitModeName'}),
    left_on='VisitMode',
    right_on='VisitModeId',
    how='left'
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Pie chart
mode_counts = transaction_with_mode['VisitModeName'].value_counts()
axes[0].pie(mode_counts.values, labels=mode_counts.index, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Visit Mode Distribution')

# Bar chart
mode_counts.plot(kind='bar', ax=axes[1], color='lightgreen', edgecolor='black')
axes[1].set_xlabel('Visit Mode')
axes[1].set_ylabel('Count')
axes[1].set_title('Visit Mode Counts')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/visit_mode_eda.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nVisit Mode Statistics:')
print(mode_counts)

## 6. Temporal Analysis

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Visits by year
year_counts = transaction['VisitYear'].value_counts().sort_index()
axes[0].plot(year_counts.index, year_counts.values, marker='o', linewidth=2, markersize=8)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Number of Visits')
axes[0].set_title('Visits Trend Over Years')
axes[0].grid(True, alpha=0.3)

# Visits by month
month_counts = transaction['VisitMonth'].value_counts().sort_index()
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
axes[1].bar(month_counts.index, month_counts.values, color='steelblue', edgecolor='black')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Number of Visits')
axes[1].set_title('Visits Distribution Across Months')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_names)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/temporal_eda.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. User Demographics Analysis

In [ ]:
# Merge with continent data
user_continent = user.merge(continent, on='ContinentId', how='left')
continent_counts = user_continent['Continent'].value_counts()

plt.figure(figsize=(12, 6))
continent_counts.plot(kind='barh', color='teal', edgecolor='black')
plt.xlabel('Number of Users')
plt.ylabel('Continent')
plt.title('User Distribution by Continent')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/continent_eda.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nUser Distribution by Continent:')
print(continent_counts)

## 8. Attraction Analysis

In [ ]:
# Merge with attractions
transaction_with_items = transaction.merge(item, on='AttractionId', how='left')
transaction_with_type = transaction_with_items.merge(attraction_type, on='AttractionTypeId', how='left')

# Top attractions
top_attractions = transaction_with_type['Attraction'].value_counts().head(10)

plt.figure(figsize=(12, 6))
top_attractions.plot(kind='barh', color='orange', edgecolor='black')
plt.xlabel('Number of Visits')
plt.ylabel('Attraction')
plt.title('Top 10 Most Visited Attractions')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/top_attractions_eda.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nTop 10 Attractions:')
print(top_attractions)

In [ ]:
# Attraction type popularity
type_counts = transaction_with_type['AttractionType'].value_counts()

plt.figure(figsize=(12, 6))
type_counts.plot(kind='bar', color='purple', edgecolor='black')
plt.xlabel('Attraction Type')
plt.ylabel('Number of Visits')
plt.title('Popularity of Attraction Types')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/type_popularity_eda.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Rating Analysis by Different Dimensions

In [ ]:
# Average rating by attraction type
avg_rating_by_type = transaction_with_type.groupby('AttractionType')['Rating'].mean().sort_values(ascending=False)

plt.figure(figsize=(12, 6))
avg_rating_by_type.plot(kind='barh', color='crimson', edgecolor='black')
plt.xlabel('Average Rating')
plt.ylabel('Attraction Type')
plt.title('Average Rating by Attraction Type')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/rating_by_type_eda.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nAverage Rating by Attraction Type:')
print(avg_rating_by_type)

In [ ]:
# Average rating by visit mode
avg_rating_by_mode = transaction_with_mode.groupby('VisitModeName')['Rating'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
avg_rating_by_mode.plot(kind='bar', color='gold', edgecolor='black')
plt.xlabel('Visit Mode')
plt.ylabel('Average Rating')
plt.title('Average Rating by Visit Mode')
plt.xticks(rotation=45, ha='right')
plt.axhline(y=transaction['Rating'].mean(), color='red', linestyle='--', label='Overall Average')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizations/rating_by_mode_eda.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nAverage Rating by Visit Mode:')
print(avg_rating_by_mode)

## 10. Correlation Analysis

In [ ]:
# Correlation matrix
numerical_cols = ['VisitYear', 'VisitMonth', 'VisitMode', 'AttractionId', 'Rating']
correlation_matrix = transaction[numerical_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.savefig('../visualizations/correlation_eda.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. User Behavior Analysis

In [ ]:
# User activity
user_activity = transaction.groupby('UserId').agg({
    'TransactionId': 'count',
    'Rating': 'mean',
    'AttractionId': 'nunique'
}).rename(columns={
    'TransactionId': 'TotalVisits',
    'Rating': 'AvgRating',
    'AttractionId': 'UniqueAttractions'
})

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Total visits
axes[0].hist(user_activity['TotalVisits'], bins=30, edgecolor='black', color='lightblue')
axes[0].set_xlabel('Visits per User')
axes[0].set_ylabel('Frequency')
axes[0].set_title('User Visit Distribution')
axes[0].grid(axis='y', alpha=0.3)

# Average rating
axes[1].hist(user_activity['AvgRating'], bins=20, edgecolor='black', color='lightcoral')
axes[1].set_xlabel('Avg Rating per User')
axes[1].set_ylabel('Frequency')
axes[1].set_title('User Rating Distribution')
axes[1].grid(axis='y', alpha=0.3)

# Unique attractions
axes[2].hist(user_activity['UniqueAttractions'], bins=30, edgecolor='black', color='lightgreen')
axes[2].set_xlabel('Unique Attractions')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Attraction Variety')
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visualizations/user_behavior_eda.png', dpi=300, bbox_inches='tight')
plt.show()

print('\nUser Activity Statistics:')
print(user_activity.describe())

## 12. Key Insights Summary

In [ ]:
print('='*80)
print('KEY INSIGHTS FROM EDA')
print('='*80)

print('\n1. DATASET OVERVIEW:')
print(f'   - Total transactions: {len(transaction):,}')
print(f'   - Unique users: {transaction["UserId"].nunique():,}')
print(f'   - Unique attractions: {transaction["AttractionId"].nunique():,}')

print('\n2. RATING INSIGHTS:')
print(f'   - Average rating: {transaction["Rating"].mean():.2f}')
print(f'   - Most common rating: {transaction["Rating"].mode()[0]}')
print(f'   - High satisfaction (4-5 stars): {(transaction["Rating"] >= 4).sum() / len(transaction) * 100:.1f}%')

print('\n3. VISIT MODE INSIGHTS:')
most_common = transaction_with_mode['VisitModeName'].value_counts().index[0]
print(f'   - Most common: {most_common}')

print('\n4. TEMPORAL INSIGHTS:')
peak_month = transaction['VisitMonth'].value_counts().index[0]
print(f'   - Peak month: Month {peak_month}')
print(f'   - Clear seasonal patterns observed')

print('\n5. ATTRACTION INSIGHTS:')
top_attr = transaction_with_type['Attraction'].value_counts().index[0]
print(f'   - Most visited: {top_attr}')

print('\n6. USER BEHAVIOR:')
print(f'   - Avg visits per user: {user_activity["TotalVisits"].mean():.2f}')
print(f'   - Avg attractions per user: {user_activity["UniqueAttractions"].mean():.2f}')

print('\n' + '='*80)
print('READY FOR MODELING!')
print('='*80)

## 13. Next Steps

### Data Preprocessing for Modeling

The insights from this EDA guide the following steps:

1. **Feature Engineering**: Create user/attraction aggregates
2. **Data Transformation**: Encode and scale features
3. **Model Development**: Train regression, classification, and recommendation models
4. **Evaluation**: Compare and select best models

### Implementation

Run the following scripts:
```bash
python src/data_preprocessing.py  # Full preprocessing pipeline
python src/models.py              # Train all models
streamlit run app.py              # Launch web application
```

---

**✅ EDA Complete!**

This analysis provides comprehensive understanding of the tourism dataset and establishes a foundation for building predictive models and recommendation systems.